# Word Order Large Experiments


## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
import re

import dotenv
import evaluate
import numpy as np
import openai
import pandas as pd
import pyrootutils
import tiktoken
from transformers import AutoTokenizer

PROJECT_ROOT = path = pyrootutils.find_root(indicator=".project-root")
DATA_DIR = PROJECT_ROOT / "data"
BATCHES_DIR = PROJECT_ROOT / "batches"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
FIGURES_DIR = PROJECT_ROOT / "paper" / "figures"
DATA_EXP_DIR = DATA_DIR / "wordorder_large_exp"
EXP_DIR = BATCHES_DIR / "wordorder_large_exp"

# create figures directory if it does not exist
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

dotenv.load_dotenv(PROJECT_ROOT / ".env")
HF_TOKEN = dotenv.get_key(PROJECT_ROOT / ".env", "HF_TOKEN") or dotenv.get_key(
    PROJECT_ROOT / ".env", "HUGGINGFACE_HUB_TOKEN"
)

## Define aesthetics

In [ ]:
import aesthetics as aes  # noqa: F401

sns = aes.sns
mtick = aes.mtick
plt = aes.plt

aes.PALETTE_METRICS
aes.PALETTE_MODELS

WORD_ORDER_PLOT_ORDER = ["SVO", "SOV", "OVS"]
WORD_ORDER_PLOT_LABELS = {
    "SVO": "SVO \u2192 SVO",
    "SOV": "SVO \u2192 SOV",
    "OVS": "SVO \u2192 OVS",
}

# Set to a concrete model name to force plotting that model.
# Leave as None to auto-select from available rows.
PLOT_MODEL = None

## Grammars & Samples

In [ ]:
EXPERIMENT_GRAMMARS = DATA_EXP_DIR / "wordorder_large_grammars.txt"

grammar_ids = []
with open(EXPERIMENT_GRAMMARS, "r") as f:
    grammar_ids = [line.strip() for line in f.readlines()]

grammar_files = [f"grammar_{gid}.json" for gid in grammar_ids]
sample_files = [f"samples_{gid}.jsonl" for gid in grammar_ids]

grammars = []
for gf in grammar_files:
    with open(DATA_EXP_DIR / gf, "r") as f:
        grammars.append(json.load(f))

samples = []
for path in sample_files:
    grammar_name = path.split("samples_")[1].split(".jsonl")[0]
    with open(DATA_EXP_DIR / path, "r") as f:
        for sample_id, line in enumerate(f):
            sample = json.loads(line)
            sample["grammar_name"] = grammar_name
            sample["sample_id"] = str(sample_id)
            samples.append(sample)

samples_df = pd.DataFrame(samples)
samples_df["input_length"] = samples_df["left_phonetic"].apply(lambda x: len(x.split()))

grammar_sizes = pd.DataFrame(
    [{"name": g["name"], "n_words": g["n_words"]} for g in grammars]
)
sns.stripplot(grammar_sizes["n_words"])

In [ ]:
sns.histplot(samples_df["depth"], discrete=True)

## Outputs

In [ ]:
openai_client = openai.OpenAI()

tokenizers = {
    "gpt-5-nano": tiktoken.encoding_for_model("gpt-5-nano"),
    "gpt-5-mini": tiktoken.encoding_for_model("gpt-5-mini"),
    "gpt-5": tiktoken.encoding_for_model("gpt-5"),
    "google/gemma-3-1b-it": AutoTokenizer.from_pretrained(
        "google/gemma-3-1b-it", token=HF_TOKEN
    ),
    "google/gemma-3-4b-it": AutoTokenizer.from_pretrained(
        "google/gemma-3-4b-it", token=HF_TOKEN
    ),
    "google/gemma-3-12b-it": AutoTokenizer.from_pretrained(
        "google/gemma-3-12b-it", token=HF_TOKEN
    ),
}


def fuzzy_model(model: str | None) -> str:
    return re.sub(r"-\d{4}-\d{2}-\d{2}$", "", model or "")

In [ ]:
outputs_glob = sorted(EXP_DIR.glob("*_output.jsonl"))
inputs_glob = [
    path
    for path in sorted(EXP_DIR.glob("inputs_*.jsonl"))
    if not path.name.endswith("_output.jsonl")
]


def parse_standard_custom_id(custom_id: str | None) -> tuple[str | None, str | None]:
    if not isinstance(custom_id, str):
        return None, None
    match = re.match(
        r"^(?P<grammar_name>[0-9a-f]+)-(?P<batch_hash>[0-9a-f]+)-sample-(?P<sample_id>\d+)$",
        custom_id,
    )
    if not match:
        return None, None
    return match.group("grammar_name"), match.group("sample_id")


output_frames = []
for path in outputs_glob:
    df = pd.read_json(path, lines=True)
    json_struct = json.loads(df.to_json(orient="records"))
    flat_df = pd.json_normalize(json_struct)
    if flat_df.empty:
        continue

    flat_df["batch_id"] = path.name.split("_output.jsonl")[0]
    flat_df["model_response"] = flat_df["response.body.choices"].apply(
        lambda x: x[0]["message"]["content"] if isinstance(x, list) and x else None
    )
    flat_df["model"] = flat_df["response.body.model"]

    usage_series = flat_df.get("response.body.usage")
    if usage_series is not None:
        flat_df["prompt_tokens"] = usage_series.apply(
            lambda x: x.get("prompt_tokens", x.get("promptTokens"))
            if isinstance(x, dict)
            else None
        )
        flat_df["completion_tokens"] = usage_series.apply(
            lambda x: x.get("completion_tokens", x.get("completionTokens"))
            if isinstance(x, dict)
            else None
        )
        flat_df["total_tokens"] = usage_series.apply(
            lambda x: x.get("total_tokens", x.get("totalTokens"))
            if isinstance(x, dict)
            else None
        )
    else:
        for col, snake, camel in [
            (
                "prompt_tokens",
                "response.body.usage.prompt_tokens",
                "response.body.usage.promptTokens",
            ),
            (
                "completion_tokens",
                "response.body.usage.completion_tokens",
                "response.body.usage.completionTokens",
            ),
            (
                "total_tokens",
                "response.body.usage.total_tokens",
                "response.body.usage.totalTokens",
            ),
        ]:
            if snake in flat_df:
                flat_df[col] = flat_df[snake]
            elif camel in flat_df:
                flat_df[col] = flat_df[camel]
            else:
                flat_df[col] = pd.NA

    output_frames.append(flat_df)

if not output_frames:
    raise FileNotFoundError(f"No batch output files found in {EXP_DIR}")

outputs_df = pd.concat(output_frames, ignore_index=True)
outputs_df = outputs_df.drop(
    [col for col in outputs_df.columns if col.startswith("response")], axis=1
).drop(columns=["error"], errors="ignore")

input_frames = []
for path in inputs_glob:
    df = pd.read_json(path, lines=True)
    json_struct = json.loads(df.to_json(orient="records"))
    flat_df = pd.json_normalize(json_struct)
    if flat_df.empty:
        continue

    flat_df["input_file"] = path.name
    flat_df["grammar_name"] = flat_df.get("body.metadata.grammar_name")
    flat_df["sample_id"] = flat_df.get("body.metadata.sample_id")
    flat_df["depth"] = pd.to_numeric(
        flat_df.get("body.metadata.depth"), errors="coerce"
    )

    parsed_ids = flat_df["custom_id"].apply(parse_standard_custom_id)
    parsed_df = pd.DataFrame(
        parsed_ids.tolist(), columns=["grammar_name_parsed", "sample_id_parsed"]
    )
    flat_df = pd.concat([flat_df.reset_index(drop=True), parsed_df], axis=1)
    flat_df["grammar_name"] = flat_df["grammar_name"].fillna(
        flat_df["grammar_name_parsed"]
    )
    flat_df["sample_id"] = flat_df["sample_id"].fillna(flat_df["sample_id_parsed"])
    flat_df["sample_id"] = flat_df["sample_id"].astype(str)
    input_frames.append(flat_df)

if not input_frames:
    raise FileNotFoundError(f"No batch input files found in {EXP_DIR}")

inputs_df = pd.concat(input_frames, ignore_index=True).drop(
    columns=[
        "body.max_completion_tokens",
        "grammar_name_parsed",
        "sample_id_parsed",
    ],
    errors="ignore",
)

merged_df = pd.merge(
    outputs_df,
    inputs_df[["custom_id", "input_file", "grammar_name", "sample_id", "depth"]],
    on="custom_id",
    how="inner",
)

merged_df = pd.merge(
    merged_df,
    samples_df[
        ["grammar_name", "sample_id", "left_phonetic", "right_phonetic", "depth"]
    ].rename(
        columns={
            "left_phonetic": "input_sentence",
            "right_phonetic": "output_sentence",
            "depth": "sample_depth",
        }
    ),
    on=["grammar_name", "sample_id"],
    how="left",
)
merged_df["depth"] = pd.to_numeric(merged_df["depth"], errors="coerce").fillna(
    pd.to_numeric(merged_df["sample_depth"], errors="coerce")
)
merged_df = merged_df.drop(columns=["sample_depth"], errors="ignore")

merged_df["fuzzy_model"] = merged_df["model"].apply(fuzzy_model)

In [ ]:
grammar_blobs = DATA_EXP_DIR.glob("grammar_*.json")
grammars = []

for path in grammar_blobs:
    with open(path, "r") as f:
        grammar = json.load(f)
        grammar_name = path.name.split("grammar_")[1].split(".json")[0]
        grammar["grammar_name"] = grammar_name
        grammar = pd.json_normalize(grammar)

        grammar["a.words"] = grammar[
            [
                "a.verbs",
                "a.nouns",
                "a.propns",
                "a.prons",
                "a.adjs",
                "a.det_def",
                "a.det_indef",
                "a.comps",
            ]
        ].apply(lambda row: sum(row, []), axis=1)
        grammar["b.words"] = grammar[
            [
                "b.verbs",
                "b.nouns",
                "b.propns",
                "b.prons",
                "b.adjs",
                "b.det_def",
                "b.det_indef",
                "b.comps",
            ]
        ].apply(lambda row: sum(row, []), axis=1)

        grammar["share_head"] = grammar["a.head_initial"] == grammar["b.head_initial"]
        grammar["share_spec"] = grammar["a.spec_initial"] == grammar["b.spec_initial"]

        def categorize(row) -> str:
            if row["share_head"] and row["share_spec"]:
                return "all-same"
            elif row["share_head"] and not row["share_spec"]:
                return "spec-diff"
            elif not row["share_head"] and row["share_spec"]:
                return "head-diff"
            else:
                return "all-diff"

        grammar["word_order_type"] = grammar.apply(categorize, axis=1)
        grammar["target_word_order"] = grammar["word_order_type"].map(
            {
                "all-same": "SVO",
                "head-diff": "SOV",
                "all-diff": "OVS",
                "spec-diff": "VOS",
            }
        )

        grammar["syllable_structure"] = grammar["a.syllable_structure"]

        grammar = grammar.drop(
            columns=[
                "a.head_initial",
                "b.head_initial",
                "a.spec_initial",
                "b.spec_initial",
                "a.pro_drop",
                "b.pro_drop",
                "a.syllable_structure",
                "b.syllable_structure",
            ]
        )

        grammars.append(grammar)

grammars_df = pd.concat(grammars, ignore_index=True)

merged_df = pd.merge(
    merged_df,
    grammars_df[
        [
            "share_head",
            "share_spec",
            "word_order_type",
            "target_word_order",
            "grammar_name",
            "syllable_structure",
            "n_rules",
            "n_words",
            "a.words",
            "b.words",
        ]
    ],
    on="grammar_name",
)

merged_df["input_tokens"] = merged_df.apply(
    lambda row: len(
        tokenizers[fuzzy_model(row["model"])].encode(row["input_sentence"])
    ),
    axis=1,
)
merged_df["output_tokens"] = merged_df.apply(
    lambda row: len(
        tokenizers[fuzzy_model(row["model"])].encode(row["output_sentence"])
    ),
    axis=1,
)

### Extract Answer

In [ ]:
answer_re = re.compile(
    r"final\s*answer\s*(?::|-|—)?\s*(?:is\s*)?([^\n]+)",
    re.DOTALL | re.IGNORECASE,
)


def extract_answer(model_response):
    matches = answer_re.findall(model_response)
    if matches:
        last_match: str = matches[-1]
        last_match = re.sub(r"[^a-zA-Z\s]", "", last_match)
        last_match = last_match.strip()
        return last_match
    else:
        return None


merged_df = merged_df.drop_duplicates(subset=["custom_id", "batch_id"])
merged_df["model_answer"] = merged_df["model_response"].apply(extract_answer)

In [ ]:
# Check how many entries there are where model_answer is null per model
null_answers = merged_df[merged_df["model_answer"].isna()]
null_answers.groupby("fuzzy_model").size()

### Compute Metrics

In [ ]:
# metrics
try:
    import sacrebleu
except ImportError:
    sacrebleu = None


def exact_match(row) -> bool:
    if row["model_answer"] is None or row["output_sentence"] is None:
        return False
    return row["model_answer"] == row["output_sentence"]


def bow_match(row) -> bool:
    if row["model_answer"] is None or row["output_sentence"] is None:
        return False
    return sorted(row["model_answer"].split()) == sorted(row["output_sentence"].split())


def edit_similarity(row) -> float:
    if row["model_answer"] is None or row["output_sentence"] is None:
        return 1.0
    from strsimpy.jaro_winkler import JaroWinkler

    jw = JaroWinkler()
    return 1 - jw.distance(row["model_answer"], row["output_sentence"])


def bleu(row) -> float:
    if sacrebleu is None:
        return np.nan
    pred = row["model_answer"] or ""
    ref = row["output_sentence"] or ""
    return sacrebleu.sentence_bleu(pred, [ref]).score / 100.0


def num_oov_words(row) -> int:
    if row["model_answer"] is None or row["output_sentence"] is None:
        return np.nan
    model_words: set[str] = set(row["model_answer"].split())
    output_words: set[str] = set(row["b.words"])
    oov_words: set[str] = model_words - output_words
    return len(oov_words)


# Metrics
merged_df["exact_match"] = merged_df.apply(exact_match, axis=1)
merged_df["bow_match"] = merged_df.apply(bow_match, axis=1)
merged_df["edit_similarity"] = merged_df.apply(edit_similarity, axis=1)
merged_df["num_oov_words"] = merged_df.apply(num_oov_words, axis=1)
merged_df["bleu"] = merged_df.apply(bleu, axis=1)

# manually calculate chrF and chrF++
preds = merged_df["model_answer"].tolist()
refs = merged_df["output_sentence"].tolist()

chrf = evaluate.load("chrf")
chrf_scores: list[float] = []
chrf_pp_scores: list[float] = []
for pred, ref in zip(preds, refs):
    if pred is None:
        pred = ""
    if ref is None:
        ref = ""
    chrf_scores.append(
        chrf.compute(
            predictions=[pred],
            references=[[ref]],
            beta=2,
            word_order=0,
        )
    )
    chrf_pp_scores.append(
        chrf.compute(
            predictions=[pred],
            references=[[ref]],
            beta=2,
            word_order=2,
        )
    )
merged_df["chrF"] = [score["score"] / 100.0 for score in chrf_scores]
merged_df["chrF++"] = [score["score"] / 100.0 for score in chrf_pp_scores]

merged_df["input_words"] = merged_df["input_sentence"].apply(lambda x: len(x.split()))
merged_df["input_words_rounded5"] = merged_df["input_words"].apply(
    lambda x: round(x / 5) * 5
)
merged_df["size"] = merged_df.apply(
    lambda row: int(row["n_rules"]) + int(row["n_words"]),
    axis=1,
)

merged_df["ttc"] = merged_df["completion_tokens"]
merged_df["ttc_binned"] = merged_df["ttc"].apply(
    lambda x: round(np.log10(x)) if x > 0 else 0
)

metrics_df = merged_df.melt(
    id_vars=[
        "model",
        "input_words_rounded5",
        "input_words",
        "custom_id",
        "model_answer",
        "output_sentence",
        "depth",
        "size",
        "ttc",
        "ttc_binned",
        "syllable_structure",
        "share_spec",
        "share_head",
        "word_order_type",
        "target_word_order",
        "input_tokens",
        "output_tokens",
    ],
    value_vars=[
        "exact_match",
        "bow_match",
        "edit_similarity",
        "num_oov_words",
        "bleu",
        "chrF",
        "chrF++",
    ],
    var_name="match_type",
    value_name="match_value",
)

metrics_df["match_type"] = metrics_df["match_type"].replace(
    {
        "exact_match": "Exact Match",
        "bow_match": "Bag of Words",
        "edit_similarity": "Edit Similarity",
        "num_oov_words": "Num OOV Words",
        "bleu": "BLEU Score",
        "chrF++": "chrF++",
        "chrF": "chrF",
    }
)

metrics_df["input_tokens_rounded"] = metrics_df["input_tokens"].apply(
    lambda x: round(x / 5) * 5
)
metrics_df["output_tokens_rounded"] = metrics_df["output_tokens"].apply(
    lambda x: round(x / 5) * 5
)
metrics_df["token_delta"] = abs(
    metrics_df["output_tokens"] - metrics_df["input_tokens"]
)
metrics_df["token_delta_norm"] = abs(
    metrics_df["output_tokens"] - metrics_df["input_tokens"]
) / metrics_df["input_tokens"].replace(0, np.nan)
metrics_df["token_delta_norm_rounded"] = metrics_df["token_delta_norm"].apply(
    lambda x: round(x, 1)
)

metrics_df["model_name"] = metrics_df["model"].apply(lambda x: fuzzy_model(x))
present_models = aes.ordered_models(metrics_df["model_name"].dropna().unique())
metrics_df["model_name"] = pd.Categorical(
    metrics_df["model_name"],
    categories=present_models,
    ordered=True,
)

In [ ]:
metrics_df["model_name"].unique()

## Plots

### Grammatical & Sentential Complexity

In [ ]:
PLOT_MODEL = "gpt-5"

In [ ]:
if len(metrics_df):
    PERF_METRICS = ["Exact Match"]

    metrics_df["input_length_quintile"] = pd.qcut(
        metrics_df["input_words"],
        q=5,
        duplicates="drop",
    )
    metrics_df["input_length_quintile_mid"] = metrics_df["input_length_quintile"].apply(
        lambda x: (x.left + x.right) / 2
    )

    selected_model = "gpt-5"
    print(f"Plotting model: {selected_model}")

    plot_source_df = metrics_df[
        (metrics_df["model_name"] == selected_model)
        & (metrics_df["match_type"].isin(PERF_METRICS))
    ].copy()
    plot_source_df["target_word_order"] = pd.Categorical(
        plot_source_df["target_word_order"],
        categories=WORD_ORDER_PLOT_ORDER,
        ordered=True,
    )

    plot_length_df = plot_source_df.copy()
    plot_size_df = plot_source_df.copy()
else:
    plot_length_df = pd.DataFrame()
    plot_size_df = pd.DataFrame()
    selected_model = None

In [ ]:
if len(plot_length_df):
    fig = plt.figure(figsize=(aes.PAPER_WIDTH_IN, aes.FIG_HEIGHT_SINGLE_ROW_IN))
    grid = fig.add_gridspec(1, 2, wspace=0.1)

    with sns.plotting_context("paper", font_scale=1, rc=aes.rcs):
        ax = fig.add_subplot(grid[0, 0])
        sns.lineplot(
            data=plot_size_df,
            x="size",
            y="match_value",
            hue="target_word_order",
            style="target_word_order",
            hue_order=WORD_ORDER_PLOT_ORDER,
            style_order=WORD_ORDER_PLOT_ORDER,
            markers=True,
            markersize=8,
            ax=ax,
            palette=aes.PALETTE_WORDORDER,
            legend=True,
            errorbar="ci",
        )
        ax.set_xlabel("Grammar size")
        ax.set_ylabel("Exact match")
        ax.set_ylim(-0.05, 1.05)
        ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
        ax.yaxis.grid(True, linestyle="--", alpha=0.5)
        ax.xaxis.set_major_formatter(aes.NICE_FORMATTER)
        ax.set_xlim(right=10_000)

        ax = fig.add_subplot(grid[0, 1])
        sns.lineplot(
            data=plot_length_df,
            x="input_length_quintile_mid",
            y="match_value",
            hue="target_word_order",
            style="target_word_order",
            hue_order=WORD_ORDER_PLOT_ORDER,
            style_order=WORD_ORDER_PLOT_ORDER,
            markers=True,
            markersize=8,
            ax=ax,
            palette=aes.PALETTE_WORDORDER,
            legend=True,
            errorbar="ci",
        )
        ax.set_xlabel("Input length (binned into 5 quantiles)")
        ax.set_ylabel("")
        ax.set_ylim(-0.05, 1.05)
        ax.yaxis.set_major_formatter(aes.PCT_FORMATTER)
        ax.yaxis.grid(True, linestyle="--", alpha=0.5)
        ax.yaxis.set_ticklabels([])

        left_legend = fig.axes[0].get_legend()
        if left_legend is not None:
            left_legend.remove()

        handles, labels = fig.axes[1].get_legend_handles_labels()
        label_to_handle = dict(zip(labels, handles))
        ordered_levels = [
            label for label in WORD_ORDER_PLOT_ORDER if label in label_to_handle
        ]
        fig.axes[1].legend(
            [label_to_handle[label] for label in ordered_levels],
            [WORD_ORDER_PLOT_LABELS[label] for label in ordered_levels],
            title="",
            bbox_to_anchor=(1.02, 1.0),
            loc="upper left",
            borderaxespad=0,
            frameon=False,
        )

    plt.subplots_adjust(left=0, bottom=0, right=1, top=1)
    aes.save_figure(FIGURES_DIR / f"{selected_model}_wordorder_large", fig=fig)

    plt.show()
else:
    print("No scored outputs available yet for plotting.")

In [ ]:
EXPORT_DIR = NOTEBOOKS_DIR / "cache" / "combined-exp"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

if len(plot_size_df) and len(plot_length_df):
    export_size_df = plot_size_df.copy()
    export_size_df["experiment"] = "wordorder_large"
    export_size_df["panel"] = "grammar_size"
    export_size_df["selected_model"] = selected_model
    export_size_df.to_csv(EXPORT_DIR / "wordorder_large_grammar_size.csv", index=False)

    export_length_df = plot_length_df.copy()
    export_length_df["experiment"] = "wordorder_large"
    export_length_df["panel"] = "string_length"
    export_length_df["selected_model"] = selected_model
    export_length_df.to_csv(
        EXPORT_DIR / "wordorder_large_string_length.csv", index=False
    )

    print(f"Saved combined-exp inputs to {EXPORT_DIR}")
else:
    print("No word-order plot data available to export.")